# Air Quality Data Analysis with Python
## Notebook 5 · Diurnal and Monthly Patterns

⏱️ About 75 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebooks 3–4 &nbsp;·&nbsp; 🏁 The capstone

Every air-quality dataset hides two clocks. The **daily clock**: traffic peaks,
cooking hours, and the atmosphere's own breathing (the boundary layer that traps
pollution near the ground at night and lifts it away by afternoon). And the
**seasonal clock**: in Nigeria, the Harmattan — dry, dusty air off the Sahara from
roughly December to February.

The two charts that reveal those clocks — the **diurnal profile** and the
**monthly average** — are the signature plots of air-quality analysis. In this
notebook you build both from scratch, then apply them across contrasting sites.

You'll learn:

* `groupby` — the single most useful idea in pandas,
* the diurnal profile: hourly means with an IQR band, on **local time**,
* weekday vs weekend comparison (two series, direct labels),
* the monthly-average chart and the Harmattan story,
* a multi-site comparison across five very different Lagos locations.

### 1. Setup (the usual recipe, plus the house style)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = "../data"  # local checkout of the course repository
if not Path(DATA).exists():  # running in Colab -> read from GitHub
    DATA = "https://raw.githubusercontent.com/rwpinder/tutorial-air-quality-data-analysis/main/data"

GRAY, BLUE, ORANGE, GREEN = "#999999", "#0072B2", "#D55E00", "#009E73"
plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#cbcbcb", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#333333", "xtick.color": "#333333", "ytick.color": "#333333",
    "legend.frameon": False,
})

lagos = pd.read_csv(f"{DATA}/lagos_pm25_recent.csv")
lagos["datetime"] = pd.to_datetime(lagos["datetime"], utc=True)
lagos = lagos.set_index("datetime").tz_convert("Africa/Lagos")
pm = lagos["pm25_value"]

def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")

print(f"Loaded {len(pm)} hours of Oshodi data, on time zone: {pm.index.tz}")

That `tz_convert("Africa/Lagos")` is about to earn its keep: a diurnal profile on
UTC would shift every peak by an hour and put the morning rush at 07:00.

### 2. groupby: split, apply, combine

`groupby` splits rows into groups, applies a calculation to each, and combines the
results. "Mean by hour of day" is one line: the index knows its own hour
(`pm.index.hour`, values 0–23), and we group the data by it:

In [ ]:
by_hour = pm.groupby(pm.index.hour)
diurnal_mean = by_hour.mean()
diurnal_mean

24 numbers — the average concentration for each hour of the day across the whole
year. The extremes:

In [ ]:
print(f"Highest: {diurnal_mean.max():.1f} µg/m³ at {diurnal_mean.idxmax():02d}:00")
print(f"Lowest:  {diurnal_mean.min():.1f} µg/m³ at {diurnal_mean.idxmin():02d}:00")

**08:00 is the peak** — the morning rush at a bus terminal, no surprise. The
afternoon minimum (~16:00) is the atmosphere at work: by mid-afternoon the heated
ground has stirred the air into a deep, well-mixed layer that dilutes emissions.

### 3. The diurnal profile chart

A mean line alone hides how variable each hour is. The Data-Explorer style adds an
**interquartile band** — the 25th–75th percentile range — behind the mean, drawn
with `fill_between` at low opacity:

In [ ]:
p25 = by_hour.quantile(0.25)
p75 = by_hour.quantile(0.75)

fig, ax = plt.subplots()
ax.fill_between(p25.index, p25, p75, color=BLUE, alpha=0.18, linewidth=0)
ax.plot(diurnal_mean.index, diurnal_mean, color=BLUE, linewidth=2.5, marker="o", markersize=4)
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, 68)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("hour of day (local time)")
ax.text(0.99, 0.05, "shaded band: middle 50% of hours", color="#555555",
        fontsize=10, ha="right", transform=ax.transAxes)
ax.annotate("08:00 — morning rush", xy=(8, diurnal_mean[8]),
            xytext=(11.5, 63), fontsize=10, color="#555555",
            arrowprops=dict(arrowstyle="-", color="#999999", linewidth=0.8))
ax.set_title("PM2.5 at Oshodi peaks with the morning rush, then the afternoon breeze cleans up");

Read the shape like an analyst: high **overnight** (shallow boundary layer traps
evening emissions), a **08:00 rush-hour peak**, a clean **afternoon minimum** as
mixing deepens, then the evening build-up begins again. One chart, the whole
daily story.

✏️ **Your turn 5.1** — Store the cleanest hour of the day in `cleanest_hour` and
its mean concentration in `cleanest_value` (hint: you saw the two methods that find
them in section 2).

In [ ]:
cleanest_hour = ...
cleanest_value = ...
print("Cleanest hour:", cleanest_hour, "at", cleanest_value, "µg/m³")

In [ ]:
check("cleanest_hour", lambda: cleanest_hour == 16,
      "diurnal_mean.idxmin() — the afternoon mixing maximum")
check("cleanest_value", lambda: 34 < cleanest_value < 40,
      "diurnal_mean.min() — around 37 µg/m³ (still 2.5x the WHO daily guideline!)")

### 4. Weekday vs weekend

Two diurnal lines answer a source question: **how much of the pattern is human
schedule?** `dayofweek` codes Monday as 0 … Sunday as 6, so `< 5` selects
weekdays. Note the direct labels at the line ends — no legend to decode:

In [ ]:
weekday = pm[pm.index.dayofweek < 5]
weekend = pm[pm.index.dayofweek >= 5]

wk_diurnal = weekday.groupby(weekday.index.hour).mean()
we_diurnal = weekend.groupby(weekend.index.hour).mean()

fig, ax = plt.subplots()
ax.plot(wk_diurnal.index, wk_diurnal, color=BLUE, linewidth=2.5)
ax.plot(we_diurnal.index, we_diurnal, color=ORANGE, linewidth=2.5)
ax.text(23.4, wk_diurnal.iloc[-1] + 1, "weekdays", color=BLUE, fontsize=11, va="center")
ax.text(23.4, we_diurnal.iloc[-1] - 2, "weekends", color=ORANGE, fontsize=11, va="center")
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
ax.set_xlim(-0.5, 27)
ax.set_ylim(0, None)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("hour of day (local time)")
ax.set_title("Oshodi barely rests: weekends run only ~8% below weekdays");

At many sites the weekend dip is dramatic. At a transport hub that never sleeps,
it's small — itself a finding about the sources.

✏️ **Your turn 5.2** — Quantify it: compute `weekend_drop_pct`, the percentage
reduction of the overall weekend mean below the overall weekday mean
(use `weekday.mean()` and `weekend.mean()` — the formula is
`(weekday - weekend) / weekday * 100`).

In [ ]:
weekend_drop_pct = ...
print("Weekend reduction:", weekend_drop_pct, "%")

In [ ]:
check("weekend_drop_pct", lambda: 4 < weekend_drop_pct < 13,
      "(weekday.mean() - weekend.mean()) / weekday.mean() * 100 — around 8%")

### 5. The monthly average — the seasonal clock

Same `groupby` idea, different clock: group by **month**. (Our file spans July
2025 – June 2026; grouping by month number folds it into a Jan–Dec seasonal view.)
Colour marks the finding: Harmattan months (Dec–Jan–Feb) in blue, the rest in
context gray:

In [ ]:
import calendar

monthly = pm.groupby(pm.index.month).mean()
colors = [BLUE if m in (12, 1, 2) else GRAY for m in monthly.index]

fig, ax = plt.subplots()
ax.bar(monthly.index, monthly, color=colors)
ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_ylabel("PM2.5 (µg/m³)")
ax.text(1, monthly[1] - 3, f"{monthly[1]:.0f}", color="white", ha="center",
        va="top", fontsize=11, fontweight="bold")
ax.set_title("Harmattan months run ~60% above the April low at Oshodi");

**The Harmattan story.** From December to February, north-easterly trade winds
carry mineral dust from the Sahara (including the Bodélé Depression, Earth's
dustiest place) across West Africa. Add dry-season burning and stagnant nights,
and December–February stand clearly above the rest — even in coastal Lagos, where
sea breezes soften the effect.

Inland is another matter.

✏️ **Your turn 5.3** — Build the monthly means for **Abuja 2024**
(`abuja_pm25_2024.csv`, reference monitor, load with the usual recipe) into
`abuja_monthly`, and compute `abuja_ratio` = worst month's mean ÷ best month's
mean. Then make the bar chart with Harmattan months accented — and give it a title
worthy of the number you find.

In [ ]:
abuja = ...
...
abuja_monthly = ...
abuja_ratio = ...
...

In [ ]:
check("abuja_monthly has 12 months", lambda: len(abuja_monthly) == 12,
      "group abuja_pm by abuja_pm.index.month and take .mean()")
check("worst month is January", lambda: abuja_monthly.idxmax() == 1,
      ".idxmax() on the monthly means")
check("abuja_ratio", lambda: 4 < abuja_ratio < 8,
      "max divided by min — Abuja's seasonal swing is around 6x")

Compare the two cities' seasonal swings: **~1.6× in coastal Lagos, ~6× in inland
Abuja.** Distance from the coast (and from the dust) is destiny.

### 6. Mini-project: five sites, one city

The file `lagos_sites_recent.csv` holds the same 12 months for **five contrasting
Lagos-area sites** in long format — a `site_name` column tells you which row
belongs to which site (this is how multi-sensor data usually arrives):

In [ ]:
sites = pd.read_csv(f"{DATA}/lagos_sites_recent.csv")
sites["datetime"] = pd.to_datetime(sites["datetime"], utc=True)
sites = sites.set_index("datetime").tz_convert("Africa/Lagos")
sites["site_name"].value_counts()

`groupby` works on any column. Mean by **site**, sorted, as a horizontal bar chart
(long labels read best horizontally), with the finding accented:

In [ ]:
site_means = sites.groupby("site_name")["pm25_value"].mean().sort_values()

fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.barh(site_means.index, site_means,
        color=[BLUE if s == "Oshodi Bus Terminal" else GRAY for s in site_means.index])
ax.grid(axis="x", color="#cbcbcb", linewidth=0.8)
ax.grid(axis="y", visible=False)
ax.set_xlabel("mean PM2.5 (µg/m³), Jul 2025 – Jun 2026")
ax.set_title("The bus terminal breathes 3x the campus air: location is exposure");

Five sensors in one city, spanning **14 to 44 µg/m³** — where you stand matters as
much as when. (Worth knowing: these are low-cost optical sensors; they can drift
and disagree with reference monitors, which is why quality control — like the
checks the AQ agent applies — matters before strong conclusions.)

✏️ **Your turn 5.4** — Pick any site *other than Oshodi* from the list above.
Filter its rows (`sites[sites["site_name"] == "..."]`), build its **diurnal
profile** into `my_diurnal` (24 hourly means of the `pm25_value` column), plot it
in the house style, and — as always — write a **finding title**. Does your site's
daily rhythm differ from the bus terminal's?

In [ ]:
my_site = sites[sites["site_name"] == ...]
my_diurnal = ...
...

In [ ]:
check("my_diurnal has 24 hours", lambda: len(my_diurnal) == 24,
      "group the site's pm25_value by .index.hour and take the mean")
check("a different site than Oshodi", lambda: float(my_diurnal.max()) < 52,
      "filter sites[sites['site_name'] == ...] with a site from the list above")

### 7. Optional (advanced): the four-panel time-variation figure

The AQ agent's Data Explorer greets every dataset with a 2×2 figure — diurnal,
weekday/weekend, monthly, and an hour-by-weekday heatmap. You've now built three
of the four panels; here they are assembled, plus the heatmap (colour represents
magnitude, using the perceptually-uniform `cividis` colormap — never rainbow):

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5))
fig.subplots_adjust(hspace=0.45, wspace=0.25)

axes[0, 0].fill_between(p25.index, p25, p75, color=BLUE, alpha=0.18, linewidth=0)
axes[0, 0].plot(diurnal_mean.index, diurnal_mean, color=BLUE, linewidth=2)
axes[0, 0].set_title("By hour", fontsize=11)

axes[0, 1].plot(wk_diurnal.index, wk_diurnal, color=BLUE, linewidth=2)
axes[0, 1].plot(we_diurnal.index, we_diurnal, color=ORANGE, linewidth=2)
axes[0, 1].text(23.3, wk_diurnal.iloc[-1], "weekday", color=BLUE, fontsize=9, va="center")
axes[0, 1].text(23.3, we_diurnal.iloc[-1] - 2, "weekend", color=ORANGE, fontsize=9, va="center")
axes[0, 1].set_xlim(-0.5, 26.5)
axes[0, 1].set_title("Weekday vs weekend", fontsize=11)

axes[1, 0].bar(monthly.index, monthly,
               color=[BLUE if m in (12, 1, 2) else GRAY for m in monthly.index])
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels([calendar.month_abbr[m][0] for m in range(1, 13)])
axes[1, 0].set_title("By month", fontsize=11)

grid = pm.groupby([pm.index.dayofweek, pm.index.hour]).mean().unstack()
mesh = axes[1, 1].pcolormesh(grid.columns, range(7), grid.values, cmap="cividis")
axes[1, 1].set_yticks(range(7))
axes[1, 1].set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=9)
axes[1, 1].grid(visible=False)
axes[1, 1].set_title("Hour x weekday", fontsize=11)
fig.colorbar(mesh, ax=axes[1, 1], label="µg/m³")

for a in (axes[0, 0], axes[0, 1]):
    a.set_xticks(range(0, 24, 6))
    a.set_xticklabels([f"{h:02d}" for h in range(0, 24, 6)])
for a in (axes[0, 0], axes[0, 1], axes[1, 0]):
    a.set_ylim(0, None)
    a.set_ylabel("PM2.5 (µg/m³)", fontsize=9)

fig.suptitle("Oshodi Bus Terminal: one year, four clocks", fontweight="bold", x=0.12, ha="left");

### 8. Optional: the same chart, interactive

The AQ agent's own charts are built with **Plotly**, which adds hover tooltips and
zooming. Here's your diurnal profile as an interactive chart — hover over the
points. (One caveat: GitHub's notebook viewer shows Plotly outputs as blank, which
is why this course teaches matplotlib as the backbone.)

In [ ]:
import plotly.express as px

pfig = px.line(x=diurnal_mean.index, y=diurnal_mean.values, markers=True,
               labels={"x": "hour of day (local time)", "y": "PM2.5 (µg/m³)"},
               title="PM2.5 at Oshodi peaks with the morning rush")
pfig.update_traces(line_color="#0072B2")
pfig.show()

### 9. Recap — and what you can now do

* `groupby` is the universal move: by hour → diurnal, by month → seasonal, by
  site → spatial comparison.
* The **diurnal profile** (mean + IQR band, local time!) reads the daily clock:
  rush hours, boundary-layer dynamics.
* The **monthly chart** reads the seasonal clock: Harmattan, ~1.6× swing in
  coastal Lagos vs ~6× in inland Abuja.
* Weekday/weekend and multi-site comparisons turn patterns into **source
  hypotheses** — with colour reserved for the finding and titles that assert it.

This is precisely the analysis the AQ agent's Data Explorer performs when it first
meets a dataset — and now you can build, read, and *question* every panel of it.

**Optional next: Notebook 6 — Live Data from the OpenAQ API**, where the data
structures from Notebook 2 come back as JSON from a real web API.